In [53]:
!pip install memory-profiler

In [54]:
from typing import Dict, List, Annotated
import numpy as np
import os
import shutil
from sklearn.cluster import MiniBatchKMeans
import pickle
import heapq
import numpy as np
from sklearn.cluster import KMeans

DB_SEED_NUMBER = 42
ELEMENT_SIZE = np.dtype(np.float32).itemsize
ID_SIZE = np.dtype(np.int32).itemsize
DIMENSION = 70

n_clusters_1 = 10000
batch_size_1 = 10000
nprobe_1 = 30

n_clusters_2 = 500
batch_size_2 = 1
nprobe_2 = 500


class VecDB:
    def __init__(self, database_file_path = "saved_db.dat", index_1_file_path = "index1.dat", cluster_1_dir_path="clusters1", cluster_2_dir_path="clusters2", index_2_file_path="index2.dat", new_db = True, db_size = None) -> None:
        self.db_path = database_file_path
        self.index_1_path = index_1_file_path
        self.index_2_path = index_2_file_path
        self.cluster_1_dir_path = cluster_1_dir_path
        self.cluster_2_dir_path = cluster_2_dir_path
        if new_db:
            if db_size is None:
                raise ValueError("You need to provide the size of the database")
            # delete the old DB file if exists
            if os.path.exists(self.db_path):
                os.remove(self.db_path)
            self.vectors = self.generate_database(db_size)

    def generate_database(self, size: int) -> None:
        rng = np.random.default_rng(DB_SEED_NUMBER)
        vectors = rng.random((size, DIMENSION), dtype=np.float32)
        self._write_vectors_to_file(vectors)
        self._build_index_1st_level()
        self._build_index_2nd_level()
        return vectors

    def _write_vectors_to_file(self, vectors: np.ndarray) -> None:
        mmap_vectors = np.memmap(self.db_path, dtype=np.float32, mode='w+', shape=vectors.shape)
        mmap_vectors[:] = vectors[:]
        mmap_vectors.flush()

    def _get_num_records(self) -> int:
        return os.path.getsize(self.db_path) // (DIMENSION * ELEMENT_SIZE)

    def insert_records(self, rows: Annotated[np.ndarray, (int, 70)]):
        num_old_records = self._get_num_records()
        num_new_records = len(rows)
        full_shape = (num_old_records + num_new_records, DIMENSION)
        mmap_vectors = np.memmap(self.db_path, dtype=np.float32, mode='r+', shape=full_shape)
        mmap_vectors[num_old_records:] = rows
        mmap_vectors.flush()
        #TODO: might change to call insert in the index, if you need
        self._build_index_1st_level()
        self._build_index_2nd_level()

    def get_one_row(self, row_num: int) -> np.ndarray:
        # This function is only load one row in memory
        try:
            offset = row_num * DIMENSION * ELEMENT_SIZE
            mmap_vector = np.memmap(self.db_path, dtype=np.float32, mode='r', shape=(1, DIMENSION), offset=offset)
            return np.array(mmap_vector[0])
        except Exception as e:
            return f"An error occurred: {e}"

    def get_n_rows(self, row_num: int, n: int) -> np.ndarray:
        # This function loads a specified number of rows starting from row_num
        try:
            offset = row_num * DIMENSION * ELEMENT_SIZE
            mmap_vector = np.memmap(self.db_path, dtype=np.float32, mode='r', shape=(n, DIMENSION), offset=offset)
            return np.array(mmap_vector)
        except Exception as e:
            return f"An error occurred: {e}"


    def get_n_random_rows(self, indices) -> np.ndarray:
        try:
            min_idx = indices.min()
            max_idx = indices.max()
            offset = min_idx * DIMENSION * ELEMENT_SIZE
            n_rows = max_idx - min_idx + 1

            mmap_vector = np.memmap(
                self.db_path,
                dtype=np.float32,
                mode='r',
                shape=(n_rows, DIMENSION),
                offset=offset
            )
            relative_indices = indices - min_idx
            return np.array(mmap_vector[relative_indices])
        except Exception as e:
            return f"An error occurred: {e}"


    def get_all_rows(self) -> np.ndarray:
        # Take care this load all the data in memory
        num_records = self._get_num_records()
        vectors = np.memmap(self.db_path, dtype=np.float32, mode='r', shape=(num_records, DIMENSION))
        return np.array(vectors)


    def retrieve(self, query: Annotated[np.ndarray, (1, DIMENSION)], top_k=5):

        if not os.path.exists(self.cluster_1_dir_path):
            raise FileNotFoundError(f"Cluster directory '{self.cluster_1_dir_path}' not found")

        if not os.path.exists(self.cluster_2_dir_path):
            raise FileNotFoundError(f"Cluster directory '{self.cluster_2_dir_path}' not found")

        with open(self.index_1_path, 'rb') as index_file:
            kmeans_1st = pickle.load(index_file)

        with open(self.index_2_path, 'rb') as index_file:
            kmeans_2nd = pickle.load(index_file)


        # getting closest nprobe2 2nd level clusters using their centroids

        second_level_distances={} #{"id":distance}
        second_level_centroids = kmeans_2nd.cluster_centers_
        second_level_labels = kmeans_2nd.labels_ # labels of 2nd level clusters (files of cluster2)


        for i in range(len(second_level_centroids)):
            distance = self._cal_score(query,second_level_centroids[i])
            second_level_distances[int(second_level_labels[i])] = distance

        sorted_second_level = dict(sorted(second_level_distances.items(), key=lambda item: item[1]))

        closest_second_level_clusters = list(sorted_second_level.keys())[:nprobe_2] #closest clusters of clusters

        # getting closest 1st level clusters
        first_level_distances = {}
        first_level_centroids = kmeans_1st.cluster_centers_
        first_level_labels = kmeans_1st.labels_ # labels of 1st level clusters (files of clusters1)

        for cluster_id in closest_second_level_clusters: # opening 2nd level clusters to access first level cluster ids
            cluster_file_path = os.path.join(self.cluster_2_dir_path, f'{cluster_id}.bin')

            if not os.path.exists(cluster_file_path):
                continue

            with open(cluster_file_path, 'rb') as f:
                while True:
                    id_bytes = f.read(ID_SIZE)
                    if not id_bytes:
                        break
                    first_level_id = np.frombuffer(id_bytes, dtype=np.int32)[0] #read first level cluster ids from 2nd level cluster files
                    distance = self._cal_score(query, first_level_centroids[first_level_id])
                    first_level_distances[first_level_id] = distance

        sorted_first_level = sorted(first_level_distances.items(), key=lambda x: x[1])
        closest_first_level_clusters = [cluster_id for cluster_id, _ in sorted_first_level[:nprobe_1]] # get closest nprobe1 1st level clusters


        nearest_neighbors = []

        # finally time to access the vectors
        for cluster_file in closest_first_level_clusters: #loop on first level cluster files, open them to access the vectors

            cluster_file_path = os.path.join(self.cluster_1_dir_path, f"{cluster_file}.bin")

            if not os.path.exists(cluster_file_path):
                continue

            ids=[]
            with open(cluster_file_path, 'rb') as f:
                while True:
                    id_bytes = f.read(ID_SIZE)

                    if not id_bytes:
                        break

                    id = np.frombuffer(id_bytes, dtype=np.int32)[0]
                    ids.append(id)

            ids_np = np.array(ids)
            vectors = self.get_n_random_rows(ids_np) # get the vectors corresponding to these ids
            for id,vector in zip(ids,vectors):
                distance = self._cal_score(query, vector)
                    # normal list
                # nearest_neighbors.append((id, vector, distance))

                    # heap queue
                if len(nearest_neighbors) < top_k:
                    heapq.heappush(nearest_neighbors, (distance,id,vector))
                else:
                    heapq.heappushpop(nearest_neighbors, (distance,id,vector))

        # heap queue
        nearest_neighbors = sorted(nearest_neighbors, key=lambda x: -x[0])

        # normal list
        # nearest_neighbors = sorted(nearest_neighbors, key=lambda x: -x[2])[:top_k]


        # Return the top-k vectors
        ids = [int(id) for _,id,_ in nearest_neighbors]
        # print("OUR IDs",ids)
        return ids


    def _cal_score(self, vec1, vec2):
        dot_product = np.dot(vec1, vec2)
        norm_vec1 = np.linalg.norm(vec1)
        norm_vec2 = np.linalg.norm(vec2)
        cosine_similarity = dot_product / (norm_vec1 * norm_vec2)
        return cosine_similarity

    def _build_index_1st_level(self):
        # Placeholder for index building logic

        # 1 000 000 / 1 000 rows = 1000 cluster
        # 10 000 000/ 1 000 rows = 10 000 cluster
        # 15 000 000/ 1 000 rows = 15 000 cluster
        # 20 000 000/ 1 000 rows = 20 000 cluster

        kmeans = MiniBatchKMeans(n_clusters=n_clusters_1, random_state=DB_SEED_NUMBER, batch_size=batch_size_1)

        for i in range(0, self._get_num_records(), batch_size_1):
            batch = self.get_n_rows(i, batch_size_1)
            kmeans.partial_fit(batch)

        with open(self.index_1_path, 'wb') as index_file:
            pickle.dump(kmeans, index_file)

        # print("Cluster centers",kmeans.cluster_centers_)
        if os.path.exists(self.cluster_1_dir_path):
            shutil.rmtree(self.cluster_1_dir_path)

        os.makedirs(self.cluster_1_dir_path, exist_ok=True)

        cluster_files = {}
        for cluster_id in range(n_clusters_1):
            file_path = os.path.join(self.cluster_1_dir_path, f'{cluster_id}.bin')
            cluster_files[cluster_id] = open(file_path, 'wb')

        try:
            for i in range(0, self._get_num_records(), batch_size_1):
                batch = self.get_n_rows(i, batch_size_1)
                labels = kmeans.predict(batch)
                ids = range(i, i + batch_size_1)

                for label, id in zip(labels, ids):
                    cluster_files[label].write(id.to_bytes(ID_SIZE, byteorder='little'))
        finally:
            for f in cluster_files.values():
                f.close()



    def _build_index_2nd_level(self):
        kmeans_2nd_level = MiniBatchKMeans(n_clusters=n_clusters_2, random_state=DB_SEED_NUMBER, batch_size=batch_size_2)

        with open(self.index_1_path, 'rb') as index_file:
            kmeans_1st_level = pickle.load(index_file)


        centroids = kmeans_1st_level.cluster_centers_
        print("centroidsssssssss: ",centroids)

        kmeans_2nd_level.fit(centroids)


        with open(self.index_2_path, 'wb') as index_file:
            pickle.dump(kmeans_2nd_level, index_file)


        if os.path.exists(self.cluster_2_dir_path):
            shutil.rmtree(self.cluster_2_dir_path)

        os.makedirs(self.cluster_2_dir_path, exist_ok=True)


        # try:
        #     labels_2nd_level = kmeans_2nd_level.predict(centroids)

        #     cluster_files = {}
        #     for cluster_id in range(n_clusters_2):
        #         file_path = os.path.join(self.cluster_dir_2_paths, f'{cluster_id}.bin')
        #         cluster_files[cluster_id] = open(file_path, 'wb')


        #         for label, id in zip(labels, ids):
        #             cluster_files[label].write(id.to_bytes(ID_SIZE, byteorder='little'))

        # finally:
        #     for f in cluster_files.values():
        #         f.close()


        # Predict second-level clusters for the first-level centroids
        first_level_lables = kmeans_1st_level.predict(centroids)
        second_level_labels = kmeans_2nd_level.predict(centroids) # second_level_labels[i] = label of centroids[i]

        # i want to convert this to a dict {"second_level_label":[1st level labels]}

        print("Unique first-level labels:", list(set(first_level_lables)))
        print("Unique second-level labels:", list(set(second_level_labels)))


        # Group first-level cluster IDs by second-level cluster
        cluster_mapping = {cluster_id: [] for cluster_id in range(n_clusters_2)}
        for first_label, second_label in zip(first_level_lables, second_level_labels):
            cluster_mapping[second_label].append(first_label)

        print("Cluster Mapping:", cluster_mapping)

        # Write each second-level cluster's associated first-level cluster IDs to files
        # for cluster_id, first_level_ids in cluster_mapping.items():
        #     file_path = os.path.join(self.cluster_2_dir_path, f'{cluster_id}.bin')
        #     with open(file_path, 'wb') as file:
        #         for first_level_id in first_level_ids:
        #             file.write(int(first_level_id).to_bytes(ID_SIZE, byteorder='little'))
        for cluster_id in cluster_mapping.keys():
            file_path = os.path.join(self.cluster_2_dir_path, f'{cluster_id}.bin')
            with open(file_path, 'wb') as file:
                for first_level_id in cluster_mapping[cluster_id]:
                    file.write(int(first_level_id).to_bytes(ID_SIZE, byteorder='little'))






In [55]:
# This snippet of code is to show you a simple evaluate for VecDB class, but the full evaluation for project on the Notebook shared with you.
import numpy as np
import time
from dataclasses import dataclass
from typing import List

@dataclass
class Result:
    run_time: float
    top_k: int
    db_ids: List[int]
    actual_ids: List[int]

def run_queries(db, queries, top_k, actual_ids, num_runs):
    """
    Run queries on the database and record results for each query.

    Parameters:
    - db: Database instance to run queries on.
    - queries: List of query vectors.
    - top_k: Number of top results to retrieve.
    - actual_ids: List of actual results to evaluate accuracy.
    - num_runs: Number of query executions to perform for testing.

    Returns:
    - List of Result
    """
    global results
    results = []
    for i in range(num_runs):
        tic = time.time()
        db_ids = db.retrieve(queries[i], top_k)
        toc = time.time()
        run_time = toc - tic
        results.append(Result(run_time, top_k, db_ids, actual_ids[i]))
    return results


def evaluate_result(results: List[Result]):
    """
    Evaluate the results based on accuracy and runtime.
    Scores are negative. So getting 0 is the best score.

    Parameters:
    - results: A list of Result objects

    Returns:
    - avg_score: The average score across all queries.
    - avg_runtime: The average runtime for all queries.
    """
    scores = []
    run_time = []
    for res in results:
        run_time.append(res.run_time)
        # case for retireving number not equal to top_k, socre will be the lowest
        if len(set(res.db_ids)) != res.top_k or len(res.db_ids) != res.top_k:
            scores.append( -1 * len(res.actual_ids) * res.top_k)
            continue
        score = 0
        for id in res.db_ids:
            try:
                ind = res.actual_ids.index(id)
                if ind > res.top_k * 3:
                    score -= ind
            except:
                score -= len(res.actual_ids)
        scores.append(score)

    return sum(scores) / len(scores), sum(run_time) / len(run_time)

# if __name__ == "__main__":
#     db = VecDB(db_size = 10**6)

#     all_db = db.get_all_rows()

#     res = run_queries(db, all_db, 5, 10)

#     print(eval(res))

In [56]:
from memory_profiler import memory_usage
import gc
import shutil

def memory_usage_run_queries(args):
    """
    Run queries and measure memory usage during the execution.

    Parameters:
    - args: Arguments to be passed to the run_queries function.

    Returns:
    - results: The results of the run_queries.
    - memory_diff: The difference in memory usage before and after running the queries.
    """
    global results
    mem_before = max(memory_usage())
    mem = memory_usage(proc=(run_queries, args, {}), interval = 1e-3)
    return results, max(mem) - mem_before


def get_actual_ids_first_k(actual_sorted_ids, k):
    """
    Retrieve the IDs from the sorted list of actual IDs.
    actual IDs has the top_k for the 20 M database but for other databases we have to remove the numbers higher than the max size of the DB.

    Parameters:
    - actual_sorted_ids: A list of lists containing the sorted actual IDs for each query.
    - k: The DB size.

    Returns:
    - List of lists containing the actual IDs for each query for this DB.
    """
    return [[id for id in actual_sorted_ids_one_q if id < k] for actual_sorted_ids_one_q in actual_sorted_ids]

db_size=10**6
db = VecDB(db_size = db_size)


centroidsssssssss:  [[0.53425384 0.3043546  0.62356645 ... 0.29493174 0.59316915 0.39694095]
 [0.7096353  0.51976365 0.4969941  ... 0.6111413  0.42755985 0.5313069 ]
 [0.6550201  0.53257513 0.41088808 ... 0.5253954  0.44271043 0.5464798 ]
 ...
 [0.3977301  0.6017199  0.66656226 ... 0.48348543 0.6783308  0.70278114]
 [0.54405195 0.6209502  0.5930094  ... 0.7040021  0.44722614 0.32913443]
 [0.5976732  0.3152493  0.295205   ... 0.32210714 0.418797   0.49652055]]
Unique first-level labels: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123

In [57]:

needed_top_k = 10000
QUERY_SEED_NUMBER = 10
rng = np.random.default_rng(QUERY_SEED_NUMBER)
query1 = rng.random((1, 70), dtype=np.float32)
query2 = rng.random((1, 70), dtype=np.float32)
query3 = rng.random((1, 70), dtype=np.float32)
queries = [query1, query2, query3]


actual_sorted_ids_1m_q1 = np.argsort(db.vectors.dot(query1.T).T / (np.linalg.norm(db.vectors, axis=1) * np.linalg.norm(query1)), axis= 1).squeeze().tolist()[::-1][:needed_top_k]
gc.collect()
actual_sorted_ids_1m_q2 = np.argsort(db.vectors.dot(query2.T).T / (np.linalg.norm(db.vectors, axis=1) * np.linalg.norm(query2)), axis= 1).squeeze().tolist()[::-1][:needed_top_k]
gc.collect()
actual_sorted_ids_1m_q3 = np.argsort(db.vectors.dot(query3.T).T / (np.linalg.norm(db.vectors, axis=1) * np.linalg.norm(query3)), axis= 1).squeeze().tolist()[::-1][:needed_top_k]
gc.collect()

actual_sorted_ids_20m = [actual_sorted_ids_1m_q1, actual_sorted_ids_1m_q2, actual_sorted_ids_1m_q3]

query_dummy = rng.random((1, 70), dtype=np.float32)

actual_ids = get_actual_ids_first_k(actual_sorted_ids_20m, db_size)
# print(actual_ids)
res, mem = memory_usage_run_queries((db, queries, 5, actual_ids, 3))
# print(res)
eval = evaluate_result(res)

shutil.make_archive('clusters1', 'zip', 'clusters1')
shutil.make_archive('clusters2', 'zip', 'clusters2')

to_print = f"\tscore\t{eval[0]}\ttime\t{eval[1]:.2f}\tRAM\t{mem:.2f} MB"
print(to_print)

	score	-50000.0	time	0.60	RAM	0.00 MB
